### Your Tasks:
1. **Task 1**: Standardize Column Headers (`snake_case`).
2. **Task 2**: Clean Text Formatting (Strip whitespace, fix casing).
3. **Task 3**: Remove Duplicate Records (Post-text cleaning).
4. **Task 4**: Standardize Categorical Values via Dictionary Mapping.
5. **Task 5**: Clean Numeric Strings & Parse Datetime Values.
6. **Task 6**: Perform Missing Value Imputation (Median & Mode).
7. **Task 7**: Detect and Cap Outliers using the Interquartile Range (IQR) Method.

In [1]:
import numpy as np
import pandas as pd

# Load raw dataset with intentional data anomalies
raw_data = {
    " Customer_ID ": ["CUST_001", "CUST_002", "CUST_001", "CUST_004", "CUST_005", "CUST_006", "CUST_007"],
    " Full Name ": ["  aarav sharma ", "BHAVNA PATEL", "  aarav sharma ", "chirag gupta", "divya REDDY ", " eshan verma", "Fiona Gallagher"],
    " Purchase_Amount ": [" $1,200.50 ", " $450.00 ", " $1,200.50 ", " $89.99 ", " $15,000.00 ", " $150.25 ", np.nan],
    " Join Date ": ["2024-01-15", "2023-11-20", "2024-01-15", "2022-05-10", "2024-03-01", "2023-08-19", "2024-02-12"],
    " Region ": ["BLR", " Bengaluru ", "BLR", "MUM", " Mumbai ", "DEL", np.nan],
    " Age ": [28, 34, 28, np.nan, 42, 29, 31]
}

df = pd.DataFrame(raw_data)

print("--- Initial Raw DataFrame ---")
print(df)

--- Initial Raw DataFrame ---
  Customer_ID        Full Name  Purchase_Amount   Join Date       Region   Age 
0     CUST_001    aarav sharma        $1,200.50   2024-01-15          BLR  28.0
1     CUST_002     BHAVNA PATEL         $450.00   2023-11-20   Bengaluru   34.0
2     CUST_001    aarav sharma        $1,200.50   2024-01-15          BLR  28.0
3     CUST_004     chirag gupta          $89.99   2022-05-10          MUM   NaN
4     CUST_005     divya REDDY       $15,000.00   2024-03-01      Mumbai   42.0
5     CUST_006      eshan verma         $150.25   2023-08-19          DEL  29.0
6     CUST_007  Fiona Gallagher              NaN  2024-02-12          NaN  31.0


---
## Task 1: Standardize Column Headers

Strip whitespace from column names, convert them to lowercase, and replace spaces with underscores.

In [2]:
# TODO: Standardize column names
# Hint: Use string methods on df.columns (e.g., df.columns.str.strip()...)

# Write your code below:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Cleaned Columns:", list(df.columns))


Cleaned Columns: ['customer_id', 'full_name', 'purchase_amount', 'join_date', 'region', 'age']


---
## Task 2: Text Normalization

1. Select text/string columns safely.
2. Strip leading/trailing whitespace across all text columns.
3. Convert `full_name` to Title Case (`"Aarav Sharma"`).

In [3]:
# TODO: Select text columns and strip whitespace
# Hint: Use select_dtypes(include=["object", "string"]) and apply(lambda col: col.str.strip())

# Write your code below:
string_cols = df.select_dtypes(include=["object", "string"]).columns
df[string_cols] = df[string_cols].apply(lambda col: col.str.strip())

# Convert full_name to Title Case
df["full_name"] = df["full_name"].str.title()

print(df[["customer_id", "full_name", "region"]])


  customer_id        full_name     region
0    CUST_001     Aarav Sharma        BLR
1    CUST_002     Bhavna Patel  Bengaluru
2    CUST_001     Aarav Sharma        BLR
3    CUST_004     Chirag Gupta        MUM
4    CUST_005      Divya Reddy     Mumbai
5    CUST_006      Eshan Verma        DEL
6    CUST_007  Fiona Gallagher        NaN


---
## Task 3: Deduplication & Structural Mapping

1. Remove exact duplicate rows (ensure whitespace was stripped first!).
2. Standardize `region` values using `region_mapping`:
   * `"BLR"` / `"Bengaluru"` $\rightarrow$ `"Bengaluru"`
   * `"MUM"` / `"Mumbai"` $\rightarrow$ `"Mumbai"`
   * `"DEL"` $\rightarrow$ `"Delhi"`

In [4]:
# TODO: Drop duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Map categorical region entries
region_mapping = {
    "BLR": "Bengaluru",
    "Bengaluru": "Bengaluru",
    "MUM": "Mumbai",
    "Mumbai": "Mumbai",
    "DEL": "Delhi"
}

# TODO: Replace values in the region column using region_mapping
df["region"] = df["region"].replace(region_mapping)

print("Region Value Counts:")
print(df["region"].value_counts(dropna=False))


Region Value Counts:
region
Bengaluru    2
Mumbai       2
Delhi        1
NaN          1
Name: count, dtype: int64


---
## Task 4: Numeric & Datetime Formatting

1. Clean `purchase_amount`: strip `$` and `,`, then convert to `float`.
2. Convert `join_date` to `datetime64`.

In [5]:
# TODO: Clean purchase_amount and cast to numeric float
# Hint: Use .str.replace() to remove '$' and ',', then pd.to_numeric()
df["purchase_amount"] = pd.to_numeric(
    df["purchase_amount"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
)

# TODO: Parse join_date as datetime
df["join_date"] = pd.to_datetime(df["join_date"])

print("Column Data Types:")
print(df.dtypes)


Column Data Types:
customer_id                object
full_name                  object
purchase_amount           float64
join_date          datetime64[ns]
region                     object
age                       float64
dtype: object


---
## Task 5: Missing Value Imputation & Outlier Capping

1. Impute missing values:
   * `age` $\rightarrow$ Median
   * `purchase_amount` $\rightarrow$ Median
   * `region` $\rightarrow$ Mode (most frequent)
2. Detect and cap outliers in `purchase_amount` using the IQR method:
   $$\text{Upper Bound} = \text{Q3} + 1.5 \times \text{IQR}$$

In [6]:
# TODO: Impute missing values using fillna()
df["age"] = df["age"].fillna(df["age"].median())
df["purchase_amount"] = df["purchase_amount"].fillna(df["purchase_amount"].median())
df["region"] = df["region"].fillna(df["region"].mode()[0])

# TODO: Calculate Q1, Q3, and IQR for purchase_amount
Q1 = df["purchase_amount"].quantile(0.25)
Q3 = df["purchase_amount"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

print(f"IQR Outlier Bounds: Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}, Upper Bound={upper_bound:.2f}")

# TODO: Cap outliers using np.where()
df["purchase_amount_capped"] = np.where(df["purchase_amount"] > upper_bound, upper_bound, df["purchase_amount"])

print("\n--- Final Cleaned DataFrame ---")
print(df)


IQR Outlier Bounds: Q1=225.19, Q3=1012.88, IQR=787.69, Upper Bound=2194.41

--- Final Cleaned DataFrame ---
  customer_id        full_name  ...   age purchase_amount_capped
0    CUST_001     Aarav Sharma  ...  28.0             1200.50000
1    CUST_002     Bhavna Patel  ...  34.0              450.00000
2    CUST_004     Chirag Gupta  ...  31.0               89.99000
3    CUST_005      Divya Reddy  ...  42.0             2194.40625
4    CUST_006      Eshan Verma  ...  29.0              150.25000
5    CUST_007  Fiona Gallagher  ...  31.0              450.00000

[6 rows x 7 columns]
